In [ ]:
# Ejecutar tras el refresh; grupos de cuenta creados en 00_setup.
dbutils.widgets.text('catalog','electrocasa_dev'); catalog=dbutils.widgets.get('catalog')
assert catalog in ('electrocasa_dev','electrocasa')
eng=f'{catalog}_ingenieria'; ana=f'{catalog}_analistas'; aud=f'{catalog}_auditoria'
# Evita divulgación de DNI y salario al conceder acceso futuro sobre la tabla Silver.
spark.sql(f"CREATE FUNCTION IF NOT EXISTS `{catalog}`.`silver`.`mask_dni`(v STRING) RETURNS STRING RETURN IF(is_account_group_member('{eng}'), v, '********')")
spark.sql(f"CREATE FUNCTION IF NOT EXISTS `{catalog}`.`silver`.`mask_salario`(v DECIMAL(18,2)) RETURNS DECIMAL(18,2) RETURN IF(is_account_group_member('{eng}'), v, CAST(NULL AS DECIMAL(18,2)))")
spark.sql(f'ALTER MATERIALIZED VIEW `{catalog}`.`silver`.`empleados_hist` ALTER COLUMN dni SET MASK `{catalog}`.`silver`.`mask_dni`')
spark.sql(f'ALTER MATERIALIZED VIEW `{catalog}`.`silver`.`empleados_hist` ALTER COLUMN salario SET MASK `{catalog}`.`silver`.`mask_salario`')
# Acceso explícito por tabla. No concedemos datos crudos o cuarentena a analistas.
for name in ('ventas_sucursal_mes','productos_desempeno','dotacion_actual','resenas_categoria','tracking_estado'):
    for group in (eng,ana,aud):
        spark.sql(f'GRANT SELECT ON TABLE `{catalog}`.`gold`.`{name}` TO `{group}`')
for name in ('ventas','productos','empleados_hist','empleados_eventos','resenas','devoluciones','tracking','cuarentena'):
    spark.sql(f'GRANT SELECT ON TABLE `{catalog}`.`silver`.`{name}` TO `{eng}`')
print('Masking y GRANTs aplicados; comprobar SHOW GRANTS y acceso con un usuario de cada grupo')
